In [ ]:
# 1. Install necessary libraries
!pip install -q bitsandbytes peft accelerate
# Only if not present in the environment
!pip install -q pandas torch transformers

In [1]:
import json
import os
import torch
import pandas as pd
import numpy as np
from pathlib import Path
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments
)
from peft import PeftModel, PeftConfig
from datasets import Dataset



2026-01-10 18:40:02.745659: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768070402.761128     393 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768070402.765879     393 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [28]:
# ==========================================
# CONFIGURATION
# ==========================================

# PATHS (Update these based on your Kaggle Input layout)
TEST_PATH = "/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/test.csv"
MC_CONS_PATH = "/kaggle/input/kdsh26-jsonl-file-characters/monte_cristo/monte_cristo_constraints_updated.jsonl"
CA_CONS_PATH = "/kaggle/input/kdsh26-jsonl-file-characters/castaways/castaways_constraints_filled.jsonl"

# Point this to where your trained LoRA adapter is saved
# Example: "/kaggle/input/your-training-notebook-name/qwen2.5-7b-books-lora-cls"
ADAPTER_MODEL_PATH = "/kaggle/input/kdsh26-qwen2-5-7b-instruct-fine-t-model-checkpoint/qwen2.5-7b-books-lora-cls/kaggle/working/qwen2.5-7b-books-lora-cls/checkpoint-5000"

BASE_MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

In [29]:
# ==========================================
# 2. REPLICATE PREPROCESSING
# ==========================================
# We must use the exact same logic to build the context as training

def load_constraints(path):
    mapping = {}
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            key = (obj["book_name"], obj["character"])
            mapping[key] = obj.get("constraints", [])
    return mapping

# Load constraints
print("Loading constraints...")
mc_constraints = load_constraints(MC_CONS_PATH)
ca_constraints = load_constraints(CA_CONS_PATH)
constraints = {**mc_constraints, **ca_constraints}

def constraint_to_sentence(book, char, c):
    dim = c["dimension"]
    val = c["value"]
    if dim == "health_state":
        return f"In {book}, {char} is {val}."
    elif dim == "family_role":
        return f"In {book}, {char} has family role: {val}."
    elif dim == "role":
        return f"In {book}, {char} is described as {val}."
    elif dim == "geographic_expertise":
        return f"{char} is familiar with {val}."
    elif dim == "criminal_history":
        return f"{char} has criminal history: {val}."
    else:
        return f"{dim}: {val}."

def build_context(book, char, max_cons=6):
    cons = constraints.get((book, char), [])
    if not cons:
        return ""
    sents = [constraint_to_sentence(book, char, c) for c in cons[:max_cons]]
    return " ".join(sents)

Loading constraints...


In [30]:
# Load Test Data
print("Loading Test Data...")
test_df = pd.read_csv(TEST_PATH)

# Apply Context Builder
test_df["context"] = test_df.apply(
    lambda r: build_context(r["book_name"], r["char"]), axis=1
)

# Construct Full Prompt (Premise + Hypothesis)
# Must match the format used in `encode_examples` during training
test_df["full_text"] = [
    f"Premise: {ctx}\nHypothesis: {claim}"
    for ctx, claim in zip(test_df["context"].fillna(""), test_df["content"])
]

print(f"Test data shape: {test_df.shape}")
print(f"Sample Input:\n{test_df.iloc[0]['full_text']}")


Loading Test Data...
Test data shape: (60, 7)
Sample Input:
Premise: 
Hypothesis: Learning that Villefort meant to denounce him to Louis XVIII, Noirtier pre-emptively handed the conspiracy dossier to a British spy—the very file the Count of Monte Cristo later acquired—thereby engineering his son’s “lawful” murder.


In [31]:
# ==========================================
# 3. LOAD MODEL & TOKENIZER
# ==========================================

print("Loading Tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading Model...")
# 4-bit Quantization Config (Same as training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# Load Base Model
base_model = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME,
    num_labels=2,
    quantization_config=bnb_config,
    device_map="auto",
    ignore_mismatched_sizes=True 
)
base_model.config.pad_token_id = tokenizer.pad_token_id
# Load the LoRA Adapter
# This merges the trained weights onto the base model on the fly
model = PeftModel.from_pretrained(base_model, ADAPTER_MODEL_PATH)
model.eval() # Set to evaluation mode

# Define Label Mappings (Must match training)
label2id = {"consistent": 0, "contradict": 1}
id2label = {0: "consistent", 1: "contradict"}


Loading Tokenizer...
Loading Model...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2.5-7B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.11/dist-packages/peft/config.py:165: UserWarning: Unexpected keyword arguments ['target_parameters'] for class LoraConfig, these are ignored. This probably means that you're loading a configuration file that was saved using a higher version of the library and additional parameters have been introduced since. It is highly recommended to upgrade the PEFT version before continuing (e.g. by running `pip install -U peft`).
  warnings.warn(


In [32]:
# ==========================================
# 4. PREPARE DATASET FOR INFERENCE
# ==========================================

def tokenize_function(examples):
    return tokenizer(
        examples["full_text"],
        padding="max_length", # Pad to max_length for consistent batching
        truncation=True,
        max_length=512
    )

# Convert pandas to HF Dataset
hf_test_ds = Dataset.from_pandas(test_df[["full_text"]])
hf_test_ds = hf_test_ds.map(tokenize_function, batched=True)

# Remove text column to send only tensors to model
hf_test_ds = hf_test_ds.remove_columns(["full_text"])


Map:   0%|          | 0/60 [00:00<?, ? examples/s]

In [33]:
# ==========================================
# 5. RUN INFERENCE
# ==========================================

# We use Trainer for easy batching and GPU management
trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="./results",
        per_device_eval_batch_size=8, # Adjust based on VRAM (8 or 16 usually safe for 7B)
        report_to="none"
    ),
    tokenizer=tokenizer
)

print("Running Prediction...")
predictions = trainer.predict(hf_test_ds)
logits = predictions.predictions

# Convert Logits to Class IDs
# logits shape: [n_samples, 2]
pred_ids = np.argmax(logits, axis=-1)

# Convert IDs to Label Strings
pred_labels = [id2label[i] for i in pred_ids]

/tmp/ipykernel_393/3889241800.py:6: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(
No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


Running Prediction...


In [34]:
# ==========================================
# 6. CREATE SUBMISSION FILE
# ==========================================

submission = pd.DataFrame({
    "id": test_df["id"],
    "label": pred_labels
})

# Save
submission.to_csv("submission.csv", index=False)
print("Submission saved to submission.csv")
print(submission.head())

Submission saved to submission.csv
    id       label
0   95  contradict
1  136  consistent
2   59  consistent
3   60  consistent
4  124  consistent


In [35]:
sub=pd.read_csv('/kaggle/working/submission.csv')
train=pd.read_csv('/kaggle/input/kharagpur-data-science-hackathon-kdsh-2026-dataset/train.csv')

In [36]:
np.sum(train['label']==sub['label'])/80

ValueError: Can only compare identically-labeled Series objects

In [37]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Prepare Lists
# Ground Truth (from the dataframe you loaded as TEST_PATH)
y_true = train["label"] 

# Predictions (from your submission dataframe)
y_pred = sub["label"]

# 2. Calculate Metrics
acc = accuracy_score(y_true, y_pred)
f1_macro = f1_score(y_true, y_pred, average="macro")
f1_weighted = f1_score(y_true, y_pred, average="weighted")

# 3. Print Results
print(f"Accuracy: {acc:.4f} ({np.sum(y_true == y_pred)}/{len(y_true)})")
print(f"F1 Score (Macro): {f1_macro:.4f}")
print(f"F1 Score (Weighted): {f1_weighted:.4f}")

print("\nClassification Report:")
print(classification_report(y_true, y_pred))

# 4. (Optional) Confusion Matrix Visualization
cm = confusion_matrix(y_true, y_pred, labels=["consistent", "contradict"])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", 
            xticklabels=["consistent", "contradict"], 
            yticklabels=["consistent", "contradict"])
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.title("Confusion Matrix")
plt.show()

ValueError: Found input variables with inconsistent numbers of samples: [80, 60]